# Preprocessing — Stack Overflow AI/ML similar-question recommender

Loads the raw crawl (`dataset/questions.csv`), keeps only AI/ML-tagged questions **first**
(so the expensive text steps below run on a small subset), then dedups, fills missing
values, applies the closed/duplicate policy, cleans text, filters to English, applies
quality filters, engineers features — and saves the clean catalog to
`dataset/questions_clean.csv`.

Training / model selection lives in **train_model.ipynb**; the Streamlit app is `app.py`.


In [1]:
!pip install -q langdetect


## Config — every threshold in one place
`domain_tags` must stay in sync with `TAGS` in `data_collection.ipynb`.

In [2]:
import os, re, json, html, unicodedata
from collections import Counter
import numpy as np
import pandas as pd

CONFIG = {
    "src_path": "dataset/questions.csv",
    "output_dir": "outputs",
    "clean_path": "dataset/questions_clean.csv",
    "domain_tags": [  # keep in sync with TAGS in data_collection.ipynb
        "python", "machine-learning", "deep-learning", "neural-network",
        "artificial-intelligence", "data-science", "nlp", "computer-vision",
        "reinforcement-learning", "tensorflow", "keras", "pytorch", "scikit-learn",
        "xgboost", "transformers", "huggingface", "langchain", "openai-api",
        "llm", "prompt-engineering", "spacy", "nltk", "opencv", "pandas",
        "numpy", "jupyter",
    ],
    "min_tag_freq": 5,          # a tag must appear >= this often to become a label
    "min_body_chars": 50,
    "min_view_count": 5,
    "catalog_answered_only": True,  # product serves solved questions
    "english_only": True,
    "split_dates": {"train_end": "2026-08-31", "val_end": "2026-10-31"},  # test = after val_end
    "tfidf": {"word_ngrams": [1, 2], "char_ngrams": [3, 5], "min_df": 2, "max_features": 100000},
    "top_k": [5, 10],
    "max_eval_queries": 500,
}

LOG = []
def log(msg):
    print(msg)
    LOG.append(str(msg))

os.makedirs(CONFIG["output_dir"], exist_ok=True)

## 0. Load

In [3]:
df = pd.read_csv(CONFIG["src_path"])
log(f"loaded {len(df)} rows from {CONFIG['src_path']}")

# type-cast
df["question_id"] = df["question_id"].astype(str)
df["creation_date"] = pd.to_datetime(df["creation_date"], utc=True)
for c in ["is_answered", "has_accepted_answer"]:
    df[c] = df[c].astype(str).str.lower().map({"true": True, "false": False})
for c in ["score", "view_count", "answer_count"]:
    df[c] = pd.to_numeric(df[c], errors="coerce").fillna(0).astype(int)

# dedup by question_id (first occurrence wins)
before = len(df)
df = df.drop_duplicates(subset="question_id").reset_index(drop=True)
log(f"dedup by question_id: {before} -> {len(df)} (removed {before - len(df)})")

loaded 528495 rows from dataset/questions.csv
dedup by question_id: 528495 -> 528495 (removed 0)


## 1. AI/ML domain filter — FIRST, before any text processing

The raw crawl is all-tags and huge (~528k rows); keeping only questions that carry at least
one AI/ML tag here shrinks the data before the slow steps (dedup, cleaning, language
detection), so the whole preprocessing runs in seconds instead of hours.


In [4]:
from collections import Counter

domain = set(CONFIG["domain_tags"])
df["tags"] = df["tags"].fillna("")
df["tag_list"] = df["tags"].str.split("|")
before = len(df)
df = df[df["tag_list"].map(lambda ts: bool(set(ts) & domain))].reset_index(drop=True)
log(f"AI/ML domain filter: {before} -> {len(df)} (dropped {before - len(df)} off-domain)")
counts = Counter(t for ts in df["tag_list"] for t in ts)
log("top tags: " + str(counts.most_common(10)))
df = df.drop(columns=["tag_list"])  # derived again in train_model.ipynb


AI/ML domain filter: 528495 -> 74126 (dropped 454369 off-domain)
top tags: [('python', 67964), ('pandas', 7064), ('python-3.x', 3920), ('dataframe', 3796), ('numpy', 2937), ('django', 2849), ('pytorch', 2276), ('matplotlib', 2109), ('machine-learning', 1908), ('tensorflow', 1853)]


## 2. Integrity — exact duplicate texts


In [5]:
# drop rows whose normalized title+body repeats (guards against re-scraped rows)
key = (df["title"].fillna("") + " " + df["body_text"].fillna("")).str.lower().str.replace(r"\s+", " ", regex=True)
before = len(df)
df = df[~key.duplicated()].reset_index(drop=True)
log(f"exact text duplicates removed: {before - len(df)}")

exact text duplicates removed: 0


## 3. Missing values


In [6]:
cols = ["title", "body_text", "code_text", "tags", "closed_reason", "duplicate_of"]
df[cols] = df[cols].fillna("")
df["has_code"] = df["code_text"].str.len() > 0   # no code is NOT invalid: text-only questions stay; only rows with neither body nor code are dropped
before = len(df)
df = df[~((df["body_text"].str.len() == 0) & (~df["has_code"]))].reset_index(drop=True)
log(f"missing values filled; rows with neither body nor code dropped: {before - len(df)}")
log(f"rows with no code: {int((~df['has_code']).sum())} ({(~df['has_code']).mean():.1%})")

missing values filled; rows with neither body nor code dropped: 0
rows with no code: 9292 (12.5%)


## 4. Closed & duplicate policy
Duplicate-closed questions leave the catalog (we keep the `duplicate_of` map for redirects);
off-topic closed questions (`Not suitable`, `Opinion-based`) are dropped too.


In [7]:
duplicate_map = df.loc[df["duplicate_of"] != "", ["question_id", "duplicate_of"]].copy()
before = len(df)
df = df[df["closed_reason"] != "Duplicate"]
log(f"dropped Duplicate-closed rows from catalog: {before - len(df)} (redirect map kept: {len(duplicate_map)} rows)")
before = len(df)
df = df[~df["closed_reason"].isin(["Not suitable for this site", "Opinion-based"])]
log(f"dropped off-topic closed rows: {before - len(df)}")
df = df.reset_index(drop=True)

dropped Duplicate-closed rows from catalog: 3082 (redirect map kept: 3083 rows)
dropped off-topic closed rows: 1150


## 5. Text cleaning
Unescape HTML entities, replace URLs/images with placeholders, normalize unicode, collapse
whitespace. We do **not** strip `<...>` — most of those are code-in-prose like `FC<T>`; only
well-known leftover HTML tags are removed.


In [8]:
URL_RE = re.compile(r"https?://\S+|www\.\S+")
IMG_RE = re.compile(r"!\[[^\]]*\]\([^)]*\)")
HTML_TAG_RE = re.compile(r"</?(?:p|div|br|ul|ol|li|h[1-6]|blockquote|table|tr|td|th|a|img|span|strong|em|code)\b[^>]*>", re.I)

def clean_text(s):
    if not isinstance(s, str):
        return ""
    s = html.unescape(s)               # &amp; -> &
    s = URL_RE.sub(" <URL> ", s)       # links -> placeholder
    s = IMG_RE.sub(" <IMG> ", s)       # markdown images -> placeholder
    s = HTML_TAG_RE.sub(" ", s)        # leftover real HTML tags only (keeps <T> generics)
    s = unicodedata.normalize("NFKC", s)
    return " ".join(s.split())         # collapse whitespace

for col in ["title", "body_text", "code_text"]:
    df[col] = df[col].map(clean_text)

# one text field for retrieval: title counted twice, then body, code, tags
df["text_all"] = (df["title"] + " " + df["title"] + " " + df["body_text"] + " "
                  + df["code_text"] + " " + df["tags"].str.replace("|", " ")).str.lower()
log("text cleaned; text_all built")
df[["title", "body_text", "code_text"]].head(3)

text cleaned; text_all built


,title,body_text,code_text
0,Pandas Groupby - Run Self Function - Then Tran...,"I need to run regression for each group, then ...","def simplereg(g, y, x): try: xvar = sm.add_con..."
1,Include old pandas dataframe after merging,I'm using this code to generate a dataframe th...,import pathlib import pandas as pd root_path =...
2,A probabilistic data structure based on flippi...,How does this data structure work and what is ...,class X: def __init__(self): self.x = 0 def pl...


## 6. Language filter (English only)


In [9]:
from langdetect import detect, DetectorFactory
DetectorFactory.seed = 0  # deterministic

def is_english(row):
    text = (row["title"] + " " + row["body_text"])[:500]
    if not text.strip():
        return True
    try:
        return detect(text) == "en"
    except Exception:
        return True

if CONFIG["english_only"]:
    before = len(df)
    df = df[df.apply(is_english, axis=1)].reset_index(drop=True)
    log(f"language filter: {before} -> {len(df)} (dropped {before - len(df)} non-English)")

language filter: 69894 -> 69880 (dropped 14 non-English)


## 7. Quality filters


In [10]:
before = len(df)
df = df[df["body_text"].str.len() >= CONFIG["min_body_chars"]]
df = df[df["view_count"] >= CONFIG["min_view_count"]]
log(f"min body chars / views: dropped {before - len(df)}")
if CONFIG["catalog_answered_only"]:
    before = len(df)
    df = df[df["is_answered"]]
    log(f"catalog_answered_only: {before} -> {len(df)} (dropped {before - len(df)} unanswered)")
df = df.reset_index(drop=True)

min body chars / views: dropped 140
catalog_answered_only: 69740 -> 36060 (dropped 33680 unanswered)


## 8. Feature engineering


In [11]:
df["log_score"] = np.log1p(df["score"].clip(lower=0))
df["log_views"] = np.log1p(df["view_count"])
df["log_answers"] = np.log1p(df["answer_count"])
df["is_closed"] = df["closed_reason"].ne("")
df["is_duplicate"] = df["duplicate_of"].ne("")
df["year_month"] = df["creation_date"].dt.strftime("%Y-%m")
df["day_of_week"] = df["creation_date"].dt.dayofweek
df["hour"] = df["creation_date"].dt.hour
df["days_since_start"] = (df["creation_date"] - df["creation_date"].min()).dt.days
df["title_len"] = df["title"].str.len()
df["body_len"] = df["body_text"].str.len()
df["code_len"] = df["code_text"].str.len()
df["code_ratio"] = df["code_len"] / (df["body_len"] + df["code_len"] + 1)

log("features done")

features done


## 9. Save the clean dataset + small sidecar artifacts

`dataset/questions_clean.csv` is the single input for `train_model.ipynb` and `app.py`.
`outputs/` keeps only small files here (duplicate map, config, log); the model artifacts
are written by `train_model.ipynb`.


In [12]:
out = CONFIG["output_dir"]
df.to_csv(CONFIG["clean_path"], index=False)
log(f"saved clean dataset: {len(df)} rows -> {CONFIG['clean_path']}")
duplicate_map.to_csv(f"{out}/duplicate_map.csv", index=False)
json.dump(CONFIG, open(f"{out}/config.json", "w"), indent=1, default=str)
open(f"{out}/preprocessing_log.txt", "w").write("\n".join(LOG))
log(f"saved duplicate_map.csv, config.json, preprocessing_log.txt to {out}/")


saved clean dataset: 36060 rows -> dataset/questions_clean.csv
saved duplicate_map.csv, config.json, preprocessing_log.txt to outputs/
